# Workshop integral de evaluación y sinulación - Julio Encalada y Sara Cruz

## Proyecto

**Robot Educativo basado en Visión Artificial para apoyar la Comprensión Lectora en Educación General Básica**

**Estudiantes:** Julio Encalada y Sara Cruz  

**Modelo de visión artificial:** YOLOv8

In [2]:
#===========================================================
# CELDA 1
# Instalación de librerías
#
# Objetivo:
# Instalar las bibliotecas necesarias para descargar el
# dataset desde Roboflow y entrenar modelos YOLOv8 mediante
# la librería Ultralytics.
#
# Resultado esperado:
# Todas las dependencias instaladas correctamente.
#===========================================================

!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.9 MB/s eta 0:00:00


In [3]:
#===========================================================
# CELDA 2
# Descarga automática del dataset
#
# Objetivo:
# Descargar desde Roboflow la versión oficial del dataset
# utilizado durante todo el proyecto experimental.
#
# Resultado esperado:
# Se crearán las carpetas:
#
# train/
# valid/
# test/
# data.yaml
#
#===========================================================

from roboflow import Roboflow
rf = Roboflow(api_key="tQF3ymHd1ARer4wUeiYU")
project = rf.workspace("julios-workspace-gq2sn").project("educabasica")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to EducaBasica-1 in yolov8:: 100%|██████████| 764/764 [00:00<00:00, 6898.29it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
#===========================================================
# CELDA 3
# Verificación del dataset
#
# Objetivo:
# Comprobar que el dataset fue descargado correctamente y
# que contiene la estructura requerida por YOLOv8.
#
# Resultado esperado:
# Visualizar las carpetas:
#
# train
# valid
# test
# data.yaml
#
#===========================================================

import os

dataset_path = dataset.location

print("Ruta del dataset:")
print(dataset_path)

print("\nContenido del dataset:")

for archivo in os.listdir(dataset_path):
    print(archivo)

Ruta del dataset:
/content/EducaBasica-1

Contenido del dataset:
README.roboflow.txt
valid
data.yaml
train
test


In [5]:
#===========================================================
# CELDA 4
# Verificación del archivo data.yaml
#
# Objetivo:
# Confirmar que YOLOv8 reconoce correctamente las cinco
# clases del proyecto y las rutas del dataset.
#
# Resultado esperado:
# Mostrar el contenido completo del archivo data.yaml.
#===========================================================

yaml_path = os.path.join(dataset_path,"data.yaml")

with open(yaml_path,"r") as f:
    print(f.read())

names:
- escuela
- hospital
- panaderia
- parque
- supermercado
nc: 5
roboflow:
  license: Private
  project: educabasica
  url: https://app.roboflow.com/julios-workspace-gq2sn/educabasica/1
  version: 1
  workspace: julios-workspace-gq2sn
test: ../test/images
train: ../train/images
val: ../valid/images



In [6]:
#===========================================================
# CELDA 5
# Carga de YOLOv8
#
# Objetivo:
# Cargar el modelo base preentrenado de YOLOv8 que será
# ajustado mediante Transfer Learning utilizando el
# dataset EducaBasica.
#
# Resultado esperado:
# Modelo cargado correctamente en memoria.
#===========================================================

from ultralytics import YOLO

model = YOLO("yolov8n.pt")

In [7]:
#===========================================================
# CELDA 6
# Configuración del proyecto
#
# Objetivo:
# Definir el nombre del experimento donde YOLO almacenará
# automáticamente modelos, métricas y gráficas.
#
# Resultado esperado:
# Creación de la carpeta:
#
# runs/detect/
#
#===========================================================

project_name = "Actividad2"

experiment_name = "Optimizacion_Hiperparametros"

In [8]:
# ==========================================================
# CELDA A1
# Diseño experimental de la optimización de hiperparámetros
#
# Objetivo:
# Definir las configuraciones experimentales utilizadas para
# analizar la influencia de distintos hiperparámetros sobre
# el desempeño del modelo YOLOv8n.
#
# Descripción:
# Se evaluarán múltiples configuraciones modificando cuatro
# hiperparámetros principales:
#
# - Learning Rate (lr0)
# - Batch Size
# - Weight Decay
# - Tamaño de imagen (imgsz)
#
# Cada configuración será entrenada utilizando el mismo
# dataset, la misma arquitectura YOLOv8n y las mismas
# particiones de entrenamiento, validación y prueba.
#
# Las métricas registradas serán:
#
# - Precision
# - Recall
# - F1-Score
# - mAP@50
# - mAP@50-95
#
# Resultado esperado:
# Obtener una base experimental que permita analizar la
# importancia de los hiperparámetros, sus efectos
# individuales y sus interacciones.
# ==========================================================

# ACTIVIDAD 2  
## Análisis, optimización de hiperparámetros y evaluación del modelo

En esta actividad se desarrollan experimentos controlados para analizar la influencia de hiperparámetros reales del modelo YOLOv8n. Los entrenamientos realizados en esta sección tienen fines de sensibilidad y optimización y no sustituyen al modelo oficial `best.pt` obtenido durante la Actividad 1.

In [15]:
# ==========================================================
# CELDA 8
# Configuración general de la Actividad 2
#
# Objetivo:
# Establecer parámetros comunes para los experimentos de
# optimización y garantizar condiciones reproducibles.
#
# Los entrenamientos de esta actividad no reemplazan el
# modelo oficial best.pt de la Actividad 1.
#
# Resultado esperado:
# Confirmación de la configuración experimental.
# ==========================================================

from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import random
import os
import glob

# ----------------------------------------------------------
# Reproducibilidad
# ----------------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ----------------------------------------------------------
# Configuración general
# ----------------------------------------------------------
MODELO_BASE = "yolov8n.pt"

# Corridas experimentales reducidas para sensibilidad
EPOCAS_EXPERIMENTALES = 80
PACIENCIA = 10

# Carpeta independiente para la Actividad 2
PROYECTO_ACTIVIDAD_2 = "/content/runs/Actividad2_Hiperparametros"

print("Configuración general cargada correctamente.")
print(f"Modelo base: {MODELO_BASE}")
print(f"Épocas por corrida: {EPOCAS_EXPERIMENTALES}")
print(f"Patience: {PACIENCIA}")
print(f"Semilla: {SEED}")
print(f"Carpeta de resultados: {PROYECTO_ACTIVIDAD_2}")

Configuración general cargada correctamente.
Modelo base: yolov8n.pt
Épocas por corrida: 80
Patience: 10
Semilla: 42
Carpeta de resultados: /content/runs/Actividad2_Hiperparametros


In [20]:
# ==========================================================
# CELDA 9
# Localización y validación del archivo data.yaml
#
# Objetivo:
# Identificar el archivo data.yaml del dataset utilizado
# previamente y verificar la definición de las cinco clases.
#
# Resultado esperado:
# Ruta del archivo data.yaml y nombres de las clases.
# ==========================================================

import yaml
from pathlib import Path

# Buscar archivos data.yaml dentro de /content
archivos_yaml = list(Path("/content").rglob("data.yaml"))

if len(archivos_yaml) == 0:
    raise FileNotFoundError(
        "No se encontró ningún archivo data.yaml. "
        "Ejecuta primero la celda de descarga del dataset."
    )

print("Archivos data.yaml encontrados:")

for i, ruta in enumerate(archivos_yaml):
    print(f"[{i}] {ruta}")

# Se utiliza inicialmente el primer archivo encontrado
DATASET_YAML = str(archivos_yaml[0])

with open(DATASET_YAML, "r", encoding="utf-8") as archivo:
    configuracion_dataset = yaml.safe_load(archivo)

nombres_clases = configuracion_dataset.get("names", [])
numero_clases = configuracion_dataset.get("nc", len(nombres_clases))

print("\nDataset seleccionado:")
print(DATASET_YAML)

print("\nNúmero de clases:")
print(numero_clases)

print("\nClases definidas:")
for indice, nombre in enumerate(nombres_clases):
    print(f"{indice}: {nombre}")

if numero_clases != 5:
    print(
        "\nADVERTENCIA: El archivo seleccionado no contiene "
        "exactamente cinco clases."
    )
else:
    print("\nValidación correcta: el dataset contiene cinco clases.")

Archivos data.yaml encontrados:
[0] /content/EducaBasica-1/data.yaml

Dataset seleccionado:
/content/EducaBasica-1/data.yaml

Número de clases:
5

Clases definidas:
0: escuela
1: hospital
2: panaderia
3: parque
4: supermercado

Validación correcta: el dataset contiene cinco clases.


In [22]:
# ==========================================================
# CELDA 10
# Diseño experimental de hiperparámetros
#
# Objetivo:
# Definir las configuraciones experimentales que serán
# evaluadas durante la optimización del modelo YOLOv8n.
#
# Resultado esperado:
# Tabla con las corridas experimentales.
# ==========================================================

import pandas as pd

experimentos = pd.DataFrame({

    "Experimento":[
        "E1","E2","E3","E4",
        "E5","E6","E7","E8",
        "E9","E10","E11","E12"
    ],

    "LearningRate":[
        0.001,
        0.001,
        0.001,
        0.005,
        0.005,
        0.005,
        0.010,
        0.010,
        0.010,
        0.005,
        0.001,
        0.010
    ],

    "Batch":[
        8,
        16,
        8,
        8,
        16,
        16,
        8,
        16,
        8,
        16,
        16,
        8
    ],

    "WeightDecay":[
        0.0001,
        0.0005,
        0.001,
        0.0001,
        0.0005,
        0.001,
        0.0001,
        0.0005,
        0.001,
        0.0001,
        0.001,
        0.0005
    ],

    "ImageSize":[
        480,
        480,
        640,
        640,
        480,
        640,
        480,
        640,
        640,
        480,
        640,
        480
    ]

})

display(experimentos)

,Experimento,LearningRate,Batch,WeightDecay,ImageSize
0,E1,0.001,8,0.0001,480
1,E2,0.001,16,0.0005,480
2,E3,0.001,8,0.0010,640
3,E4,0.005,8,0.0001,640
4,E5,0.005,16,0.0005,480
5,E6,0.005,16,0.0010,640
6,E7,0.010,8,0.0001,480
7,E8,0.010,16,0.0005,640
8,E9,0.010,8,0.0010,640
9,E10,0.005,16,0.0001,480


In [24]:
# ==========================================================
# CELDA 11
# Instalación de herramientas para optimización y análisis
#
# Objetivo:
# Instalar Optuna y las librerías necesarias para ejecutar
# la optimización de hiperparámetros, calcular su importancia
# y generar gráficos de dependencia e interacción.
#
# Resultado esperado:
# Mensaje de instalación completada correctamente.
# ==========================================================

!pip install -q optuna plotly kaleido scikit-learn

print("Optuna y las herramientas de análisis se instalaron correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.7 MB/s eta 0:00:00
Optuna y las herramientas de análisis se instalaron correctamente.


In [26]:
# ==========================================================
# CELDA 12
# Función general de entrenamiento y evaluación para Optuna
#
# Objetivo:
# Definir una función reutilizable que entrene YOLOv8n con
# una configuración de hiperparámetros y recupere las
# métricas obtenidas sobre el conjunto de validación.
#
# Importante:
# - Se fija AdamW para garantizar que YOLO respete lr0.
# - Cada ensayo comienza desde yolov8n.pt.
# - La métrica objetivo será mAP50-95.
# - Esta celda únicamente define la función.
#
# Resultado esperado:
# Mensaje que confirme que la función fue definida.
# ==========================================================

from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import torch
import time
import os


def entrenar_y_evaluar_yolo(
    nombre_experimento,
    lr0,
    batch,
    weight_decay,
    imgsz,
    epochs=EPOCAS_EXPERIMENTALES
):
    """
    Entrena YOLOv8n con una configuración determinada y
    devuelve las métricas generales obtenidas sobre validación.

    Parámetros
    ----------
    nombre_experimento : str
        Nombre único asignado al ensayo.

    lr0 : float
        Tasa de aprendizaje inicial.

    batch : int
        Tamaño del lote.

    weight_decay : float
        Coeficiente de regularización L2.

    imgsz : int
        Tamaño de entrada de la imagen.

    epochs : int
        Número máximo de épocas.

    Retorna
    -------
    dict
        Hiperparámetros, métricas, duración y rutas generadas.
    """

    print("\n" + "=" * 70)
    print(f"INICIANDO ENSAYO: {nombre_experimento}")
    print("=" * 70)

    print(f"Learning rate inicial: {lr0}")
    print(f"Batch size: {batch}")
    print(f"Weight decay: {weight_decay}")
    print(f"Tamaño de imagen: {imgsz}")
    print(f"Épocas máximas: {epochs}")
    print(f"Patience: {PACIENCIA}")
    print("Optimizador: AdamW")

    # ------------------------------------------------------
    # Liberación de memoria antes del entrenamiento
    # ------------------------------------------------------
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        dispositivo = 0
        print("Dispositivo: GPU")
    else:
        dispositivo = "cpu"
        print("Dispositivo: CPU")

    # ------------------------------------------------------
    # Inicio del cronómetro
    # ------------------------------------------------------
    tiempo_inicio = time.time()

    # Cada ensayo parte de los mismos pesos preentrenados
    modelo = YOLO(MODELO_BASE)

    try:
        # --------------------------------------------------
        # Entrenamiento
        # --------------------------------------------------
        resultado_entrenamiento = modelo.train(
            data=DATASET_YAML,
            epochs=int(epochs),
            patience=int(PACIENCIA),
            imgsz=int(imgsz),
            batch=int(batch),

            # Optimización
            optimizer="AdamW",
            lr0=float(lr0),
            weight_decay=float(weight_decay),

            # Reproducibilidad
            seed=int(SEED),
            deterministic=True,

            # Ejecución
            device=dispositivo,
            workers=2,
            pretrained=True,

            # Almacenamiento
            project=PROYECTO_ACTIVIDAD_2,
            name=nombre_experimento,
            exist_ok=False,

            # Salidas
            plots=False,
            verbose=False
        )

        # --------------------------------------------------
        # Localización del mejor modelo
        # --------------------------------------------------
        carpeta_experimento = Path(resultado_entrenamiento.save_dir)
        ruta_mejor_modelo = (
            carpeta_experimento / "weights" / "best.pt"
        )

        if not ruta_mejor_modelo.exists():
            raise FileNotFoundError(
                "No se encontró el archivo best.pt en: "
                f"{ruta_mejor_modelo}"
            )

        # --------------------------------------------------
        # Validación independiente del mejor peso
        # --------------------------------------------------
        modelo_validacion = YOLO(str(ruta_mejor_modelo))

        metricas = modelo_validacion.val(
            data=DATASET_YAML,
            imgsz=int(imgsz),
            batch=int(batch),
            device=dispositivo,
            workers=2,
            plots=False,
            verbose=False
        )

        # --------------------------------------------------
        # Extracción de métricas generales
        # --------------------------------------------------
        precision = float(metricas.box.mp)
        recall = float(metricas.box.mr)
        map50 = float(metricas.box.map50)
        map50_95 = float(metricas.box.map)

        if precision + recall > 0:
            f1 = (
                2 * precision * recall
                / (precision + recall)
            )
        else:
            f1 = 0.0

        tiempo_total_min = (
            time.time() - tiempo_inicio
        ) / 60

        # Época final registrada en results.csv
        ruta_results_csv = (
            carpeta_experimento / "results.csv"
        )

        epocas_ejecutadas = np.nan

        if ruta_results_csv.exists():
            resultados_epocas = pd.read_csv(
                ruta_results_csv
            )
            epocas_ejecutadas = len(
                resultados_epocas
            )

        resultado = {
            "Experimento": nombre_experimento,
            "LearningRate": float(lr0),
            "Batch": int(batch),
            "WeightDecay": float(weight_decay),
            "ImageSize": int(imgsz),
            "Optimizer": "AdamW",
            "EpochsMax": int(epochs),
            "EpochsExecuted": epocas_ejecutadas,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "mAP50": map50,
            "mAP50_95": map50_95,
            "TiempoMinutos": tiempo_total_min,
            "RutaResultados": str(
                carpeta_experimento
            ),
            "RutaBestPT": str(
                ruta_mejor_modelo
            ),
            "Estado": "COMPLETADO"
        }

        print("\nEnsayo finalizado correctamente.")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1: {f1:.4f}")
        print(f"mAP50: {map50:.4f}")
        print(f"mAP50-95: {map50_95:.4f}")
        print(
            "Duración: "
            f"{tiempo_total_min:.2f} minutos"
        )

        del modelo_validacion

        return resultado

    except Exception as error:

        tiempo_total_min = (
            time.time() - tiempo_inicio
        ) / 60

        print("\nERROR DURANTE EL ENSAYO")
        print(str(error))

        return {
            "Experimento": nombre_experimento,
            "LearningRate": float(lr0),
            "Batch": int(batch),
            "WeightDecay": float(weight_decay),
            "ImageSize": int(imgsz),
            "Optimizer": "AdamW",
            "EpochsMax": int(epochs),
            "EpochsExecuted": np.nan,
            "Precision": np.nan,
            "Recall": np.nan,
            "F1": np.nan,
            "mAP50": np.nan,
            "mAP50_95": np.nan,
            "TiempoMinutos": tiempo_total_min,
            "RutaResultados": "",
            "RutaBestPT": "",
            "Estado": f"ERROR: {str(error)}"
        }

    finally:

        # --------------------------------------------------
        # Liberación de memoria después de cada ensayo
        # --------------------------------------------------
        if "modelo" in locals():
            del modelo

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


print("Función entrenar_y_evaluar_yolo definida correctamente.")
print("La Celda 12 no ejecutó ningún entrenamiento.")

Función entrenar_y_evaluar_yolo definida correctamente.
La Celda 12 no ejecutó ningún entrenamiento.


In [28]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU.")

CUDA disponible: False
No se detectó GPU.


In [27]:
# ==========================================================
# CELDA 13
# Ensayo piloto corregido con AdamW
#
# Objetivo:
# Comprobar que la función general definida en la Celda 12
# entrena correctamente YOLOv8n, respeta los hiperparámetros
# indicados y recupera automáticamente las métricas.
#
# Importante:
# - Se utilizan solo 3 épocas.
# - Este ensayo NO forma parte de los resultados oficiales.
# - No se incluirá en las tablas ni figuras del informe.
#
# Resultado esperado:
# Tabla técnica con Precision, Recall, F1, mAP50 y mAP50-95.
# ==========================================================

resultado_piloto = entrenar_y_evaluar_yolo(
    nombre_experimento="PILOTO_ADAMW",
    lr0=0.001,
    batch=8,
    weight_decay=0.0001,
    imgsz=480,
    epochs=3
)

tabla_piloto = pd.DataFrame([resultado_piloto])

columnas_piloto = [
    "Experimento",
    "LearningRate",
    "Batch",
    "WeightDecay",
    "ImageSize",
    "Optimizer",
    "EpochsExecuted",
    "Precision",
    "Recall",
    "F1",
    "mAP50",
    "mAP50_95",
    "TiempoMinutos",
    "Estado"
]

print("\nRESULTADO DEL ENSAYO PILOTO\n")

display(
    tabla_piloto[columnas_piloto].round({
        "LearningRate": 4,
        "WeightDecay": 5,
        "Precision": 4,
        "Recall": 4,
        "F1": 4,
        "mAP50": 4,
        "mAP50_95": 4,
        "TiempoMinutos": 2
    })
)

print("\nRuta del mejor modelo:")
print(resultado_piloto["RutaBestPT"])


INICIANDO ENSAYO: PILOTO_ADAMW
Learning rate inicial: 0.001
Batch size: 8
Weight decay: 0.0001
Tamaño de imagen: 480
Épocas máximas: 3
Patience: 10
Optimizador: AdamW
Dispositivo: CPU
Ultralytics 8.4.97 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/EducaBasica-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4

KeyboardInterrupt: 